In [ ]:
import pandas as pd
import openai
import re
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata

# ==========================================
# 1. API & MODEL CONFIGURATION
# ==========================================
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = "sk-or-v1-YOUR-KEY-HERE"

# Using standard synchronous OpenAI client
client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL_NAME = "deepseek/deepseek-v4-flash"

# Controls concurrency via threads (10 threads = 10 parallel API calls)
MAX_WORKERS = 10

# ==========================================
# 2. SELECT DATASET (Uncomment ONE at a time)
# ==========================================
# INPUT_FILE, OUTPUT_FILE = "bangla_med_qa_correct.csv", "ds_evaluated_bangla_med_qa_correct.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v1.csv", "ds_evaluated_wrong_answers_v1.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v2.csv", "ds_evaluated_wrong_answers_v2.csv"
INPUT_FILE, OUTPUT_FILE = "wrong_answers_v3.csv", "ds_evaluated_wrong_answers_v3.csv"

# ==========================================
# 3. EVALUATION & PARSING FUNCTIONS
# ==========================================
def evaluate_pair(question, proposed_answer, retries=4):
    prompt = f"""You are a strict medical accuracy evaluator. Decide whether the provided model answer is correct for the question.
Only reply with a single digit: 1 or 0. No explanation, no punctuation, no extra text.
1 means the answer is factually correct and medically supported.
0 means the answer is hallucinated.

Question: {question}
Model answer: {proposed_answer}
Answer now:"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                # Exponential backoff on rate limits or API drops
                time.sleep(2 * (attempt + 1))
            else:
                return f"ERROR: {str(e)}"

def parse_binary_score(raw_text):
    if str(raw_text).startswith("ERROR:"):
        return 0
    clean_text = re.sub(r'<.*?>', '', str(raw_text)).strip()
    match = re.search(r'\b(0|1)\b', clean_text)
    if match:
        return int(match.group(1))
    return 1 if clean_text == "1" else 0

# ==========================================
# 4. WORKER FUNCTION & THREADED EXECUTION
# ==========================================
def process_row(idx, row, total_rows):
    question = row.get('question', '')
    answer = row.get('answer', '')

    raw_output = evaluate_pair(question, answer)
    score = parse_binary_score(raw_output)

    print(f"[{idx + 1}/{total_rows}] Score: {score} | Raw: '{raw_output}'")
    return idx, raw_output, score

def run_pipeline():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: File '{INPUT_FILE}' not found in current directory.")
        return

    print(f"⚡ Starting Threaded Evaluation on '{INPUT_FILE}' using model: {MODEL_NAME}")
    print("=" * 60)

    df = pd.read_csv(INPUT_FILE)
    total_rows = len(df)

    results = []

    # ThreadPoolExecutor submits all rows and handles 10 parallel execution slots
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_row, idx, row, total_rows)
            for idx, row in df.iterrows()
        ]

        for future in as_completed(futures):
            results.append(future.result())

    # Sort results back to match original row order
    results.sort(key=lambda x: x[0])

    df['model_raw_response'] = [r[1] for r in results]
    df['isCorrect'] = [r[2] for r in results]

    # Metrics calculation
    is_wrong_dataset = "wrong" in INPUT_FILE.lower()
    if is_wrong_dataset:
        evaluator_correct_count = (df['isCorrect'] == 0).sum()
    else:
        evaluator_correct_count = (df['isCorrect'] == 1).sum()

    evaluator_accuracy = (evaluator_correct_count / total_rows) * 100

    df.to_csv(OUTPUT_FILE, index=False)
    print("=" * 60)
    print(f"✅ Fast Processing Complete! Output saved to: '{OUTPUT_FILE}'")

    if is_wrong_dataset:
        print(f"   Evaluator Accuracy (Successfully flagged 0s): {evaluator_correct_count} / {total_rows} ({evaluator_accuracy:.2f}%)")
    else:
        print(f"   Evaluator Accuracy (Successfully flagged 1s): {evaluator_correct_count} / {total_rows} ({evaluator_accuracy:.2f}%)")

# Run directly without any 'await'
if __name__ == "__main__":
    run_pipeline()